# Local search (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/local-search_python.ipynb)

Executable notebook of the chapter “Local search” (part “Heuristic algorithms” of the course *Combinatorial Optimization*): it contains the local search of the chapter for the knapsack problem, the bin packing problem, the TSP and $P\,||\,C_{\max}$: the generic scheme, variable neighbourhood descent, the steps of every neighbourhood and the multi-start 2-opt, written in Python and run on every example of the chapter and on every instance of the exercises, checking each result against the expected one.

The full chapter — pseudo-code, examples, proofs — is [on the website](https://fabiofurini.github.io/combinatorial-optimization/heuristic-algorithms/local-search/). The same notebook is also available in [C++](https://colab.research.google.com/github/fabiofurini/combinatorial-optimization/blob/main/notebooks/local-search_cpp.ipynb).

**How to run it.** On Colab click the badge and then *Runtime → Run all*, or run the cells one at a time with `Shift+Enter`: the cells must be run in order, since each one uses the functions defined above it. Nothing to install: only the Python standard library is used.

**Indices.** In the notes items, jobs, cities, machines and bins are numbered from 1, here from 0: the algorithmic logic is unchanged.

## Starting solutions, random numbers and evaluation

The starting solutions (recalled from the chapter on constructive heuristics), the Park–Miller generator, the random solutions and the evaluation functions. The constants `FIRST`, `BEST` select the rule, `CMAX`, `SQ`, `NB` the objective.

In [1]:
# local search for the KP, the BPP, the TSP and P||Cmax
# items, jobs, cities, machines and bins are indexed 0, 1, ...
# every step function looks for an improving neighbour of the current solution:
# if it finds one, it applies the move to the solution (in place) and returns True; otherwise it returns False
# rule = FIRST (first improvement) or BEST (best improvement)

import math

FIRST, BEST = 0, 1                     # first improvement, best improvement
CMAX, SQ, NB = 0, 1, 2                 # objective: makespan, sum of squared loads, number of bins

In C++ the seed is a `long long&` updated by the generator. Python integers cannot be passed
by reference, so here the seed is a one-element list: `seed = [1]`, and the generator updates `seed[0]`.

In [2]:
def park_miller(seed):
    """Park-Miller minimal standard generator: seed = 16807 * seed mod (2^31 - 1).
    Returns u = seed / (2^31 - 1), a pseudo-random number in (0, 1); seed is a list [value], updated in place."""
    seed[0] = (16807 * seed[0]) % 2147483647
    return seed[0] / 2147483647.0


def random_int(seed, k):
    """Pseudo-random integer in {0, 1, ..., k-1}: floor(u * k), computed in exact integer arithmetic."""
    park_miller(seed)
    return (seed[0] * k) // 2147483647


def random_tour(n, seed):
    """Random tour with city 0 in the first position (Fisher-Yates shuffle of the positions 1..n-1)."""
    pi = list(range(n))
    for i in range(n - 1, 1, -1):
        j = 1 + random_int(seed, i)                     # j in {1, ..., i}
        pi[i], pi[j] = pi[j], pi[i]
    return pi


def random_permutation(n, seed):
    """Random permutation of 0..n-1 (Fisher-Yates shuffle)."""
    a = list(range(n))
    for i in range(n - 1, 0, -1):
        j = random_int(seed, i + 1)                     # j in {0, ..., i}
        a[i], a[j] = a[j], a[i]
    return a


def random_assignment(n, m, seed):
    """Random assignment of n jobs to m machines."""
    mach = [0] * n
    for j in range(n):
        mach[j] = random_int(seed, m)
    return mach

In [3]:
def tour_length(n, d, pi):
    """Length of the tour pi[0..n-1]."""
    length = 0
    for k in range(n):
        length = length + d[pi[k]][pi[(k + 1) % n]]
    return length


def nearest_neighbour(n, d, s):
    """Nearest neighbour heuristic from city s (ties broken by the smallest index);
    the tour is returned rotated so that it starts from city 0."""
    tour = [s]
    visited = [False] * n
    visited[s] = True
    while len(tour) < n:
        last, nxt = tour[-1], -1
        for j in range(n):
            if not visited[j] and (nxt == -1 or d[last][j] < d[last][nxt]):
                nxt = j
        tour.append(nxt)
        visited[nxt] = True
    pi = []
    k0 = 0
    while tour[k0] != 0:
        k0 += 1
    for k in range(n):
        pi.append(tour[(k0 + k) % n])
    return pi

In [4]:
def lpt(n, m, t):
    """Longest processing time first: the jobs, by non-increasing time (ties by index),
    are assigned to the least loaded machine (ties by index). Returns mach[j], the machine of job j."""
    order = list(range(n))
    for a in range(n):                                  # insertion sort, stable
        b = a
        while b > 0 and t[order[b]] > t[order[b - 1]]:
            order[b], order[b - 1] = order[b - 1], order[b]
            b -= 1
    L = [0] * m
    mach = [0] * n
    for a in range(n):
        j, h = order[a], 0
        for q in range(1, m):
            if L[q] < L[h]:
                h = q
        mach[j] = h
        L[h] = L[h] + t[j]
    return mach

In [5]:
def first_fit(n, w, c, order):
    """First fit, with the items examined in the given order; bins are labelled 0, 1, ... in opening order.
    Returns bin[i], the bin of item i."""
    L = []
    bin = [0] * n
    for a in range(n):
        i, b = order[a], 0
        while b < len(L) and L[b] + w[i] > c:
            b += 1
        if b == len(L):
            L.append(0)                                 # open a new bin
        L[b] = L[b] + w[i]
        bin[i] = b
    return bin


def next_fit(n, w, c):
    """Next fit: only the last opened bin is considered. Returns bin[i], the bin of item i."""
    bin = [0] * n
    b, load = 0, 0
    for i in range(n):
        if load + w[i] > c:                             # close the bin, open a new one
            b = b + 1
            load = 0
        load = load + w[i]
        bin[i] = b
    return bin

In [6]:
def greedy_kp(n, p, w, c, order):
    """Greedy algorithm for the KP, with the items examined in the given order. Returns x."""
    x = [0] * n
    rc = c                                              # residual capacity
    for a in range(n):
        i = order[a]
        if w[i] <= rc:
            x[i] = 1
            rc = rc - w[i]
    return x

In [7]:
def loads(n, k, assign, size):
    """Loads of k bins (or machines): L[b] = sum of size[i] over the i with assign[i] = b."""
    L = [0] * k
    for i in range(n):
        L[assign[i]] = L[assign[i]] + size[i]
    return L


def number_of_bins(n, bin):
    """Number of non-empty bins."""
    used = [False] * n
    nb = 0
    for i in range(n):
        if not used[bin[i]]:
            used[bin[i]] = True
            nb = nb + 1
    return nb


def euclidean_distances(pts):
    """Euclidean distances between points with integer coordinates, rounded to the nearest integer."""
    n = len(pts)
    d = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            dx = float(pts[i][0] - pts[j][0])
            dy = float(pts[i][1] - pts[j][1])
            d[i][j] = math.floor(math.sqrt(dx * dx + dy * dy) + 0.5)   # llround: halves away from zero
    return d

## Local search and variable neighbourhood descent

The generic **local search** and the **variable neighbourhood descent** receive the steps as functions without arguments, which act on the current solution. Every step looks for an improving neighbour: if it finds one, it applies the move and returns true.

In [8]:
def local_search(step):
    """Generic local search: step() looks for an improving neighbour of the current solution
    and, if it finds one, moves to it and returns True. Returns the number of moves performed."""
    moves = 0                           # number of moves performed
    while step():                       # an improving neighbour was found and accepted
        moves = moves + 1
    return moves                        # the current solution is a local optimum

In [9]:
def vnd(steps):
    """Variable neighbourhood descent: steps[0], ..., steps[K-1] explore the neighbourhoods N_1, ..., N_K;
    after every improving move the search restarts from the first neighbourhood.
    Returns the number of moves performed."""
    K = len(steps)                      # number of neighbourhoods
    k = 0                               # current neighbourhood
    moves = 0                           # number of moves performed
    while k < K:
        if steps[k]():                  # improving neighbour found in N_k, and accepted
            moves = moves + 1
            k = 0                       # back to the first neighbourhood
        else:
            k = k + 1                   # local optimum for N_k: try the next neighbourhood
    return moves                        # local optimum for all the K neighbourhoods

As in C++, the step functions take no arguments: the data and the current solution are captured
by a closure, e.g. `local_search(lambda: two_opt_step(n, d, pi, BEST))`; the step modifies `pi` in place.

## TSP: 2-opt and Or-opt

The **2-opt step** and the **Or-opt step**; city $0$ stays in the first position of the tour.

In [10]:
def two_opt_step(n, d, pi, rule):
    """One step of the 2-opt local search on the tour pi[0..n-1] (pi[0] never moves).
    The move (i, j) removes the edges {pi[i], pi[i+1]} and {pi[j], pi[j+1]},
    reverses the path pi[i+1..j] and adds the edges {pi[i], pi[j]} and {pi[i+1], pi[j+1]}."""
    best_delta = 0                      # best length change found so far
    bi, bj = -1, -1                     # best move found so far
    stop = False
    i = 0
    while i <= n - 3 and not stop:
        for j in range(i + 2, n):
            if i == 0 and j == n - 1:
                continue                                  # the two edges are adjacent
            u1, v1 = pi[i], pi[i + 1]                     # first removed edge
            u2, v2 = pi[j], pi[(j + 1) % n]               # second removed edge
            delta = d[u1][u2] + d[v1][v2] - d[u1][v1] - d[u2][v2]
            if delta < best_delta:
                best_delta, bi, bj = delta, i, j          # improving move
                if rule == FIRST:
                    stop = True
                    break
        i += 1
    if bi == -1:
        return False                    # no improving move: local optimum

    a, b = bi + 1, bj                   # reverse pi[bi+1..bj]
    while a < b:
        pi[a], pi[b] = pi[b], pi[a]
        a += 1
        b -= 1
    return True

In [11]:
def or_opt_step(n, d, pi, rule):
    """One step of the Or-opt local search on the tour pi[0..n-1] (pi[0] never moves).
    The move (l, i, k) takes the segment pi[i..i+l-1] of l = 1, 2, 3 consecutive cities
    and inserts it, with the same orientation, between pi[k] and pi[k+1]."""
    best_delta = 0                      # best length change found so far
    bl, bi, bk = -1, -1, -1             # best move found so far
    stop = False
    l = 1
    while l <= 3 and not stop:
        i = 1
        while i + l - 1 <= n - 1 and not stop:
            p, s1 = pi[i - 1], pi[i]                      # predecessor and first city of the segment
            sl, q = pi[i + l - 1], pi[(i + l) % n]        # last city of the segment and successor
            for k in range(n):
                if k >= i - 1 and k <= i + l - 1:
                    continue                              # same position, or inside the segment
                a, b = pi[k], pi[(k + 1) % n]             # edge where the segment is inserted
                delta = d[p][q] + d[a][s1] + d[sl][b] - d[p][s1] - d[sl][q] - d[a][b]
                if delta < best_delta:
                    best_delta, bl, bi, bk = delta, l, i, k   # improving move
                    if rule == FIRST:
                        stop = True
                        break
            i += 1
        l += 1
    if bl == -1:
        return False                    # no improving move: local optimum

    newpi = []                          # the new tour
    for k in range(n):
        if k >= bi and k <= bi + bl - 1:
            continue                                      # the segment is copied after pi[bk]
        newpi.append(pi[k])
        if k == bk:
            for r in range(bi, bi + bl):
                newpi.append(pi[r])
    pi[:] = newpi                       # overwrite the tour in place
    return True

## $P\,||\,C_{\max}$: move and swap

The change $F$ of the objective of $P\,||\,C_{\max}$, for the makespan and for the sum of the squared loads, and the **move** and **swap steps**.

In [12]:
def objective_change(m, L, a, b, delta, obj):
    """Change of the objective of P||Cmax when the time delta is transferred from machine a to machine b
    (a move transfers delta = t_j, a swap transfers delta = t_j - t_k).
    obj = CMAX: change of the makespan; obj = SQ: change of the sum of the squared loads."""
    if obj == SQ:
        return 2 * delta * (L[b] - L[a] + delta)

    Cmax, newC = 0, 0                                   # makespan before and after
    for q in range(m):
        if L[q] > Cmax:
            Cmax = L[q]
        newL = L[q]
        if q == a:
            newL = L[q] - delta
        if q == b:
            newL = L[q] + delta
        if newL > newC:
            newC = newL
    return newC - Cmax

In [13]:
def pcmax_move_step(n, m, t, mach, L, obj, rule):
    """One step of the move local search for P||Cmax.
    mach[j] is the machine of job j, L[h] the load of machine h (both updated in place);
    obj = CMAX (makespan) or SQ (sum of the squared loads); both are minimized."""
    best_delta = 0                                      # best change of the objective found so far
    bj, bh = -1, -1                                     # best move found so far
    stop = False
    j = 0
    while j < n and not stop:
        a = mach[j]                                     # machine of job j
        for h in range(m):
            if h == a:
                continue
            delta = objective_change(m, L, a, h, t[j], obj)   # job j from a to h
            if delta < best_delta:
                best_delta, bj, bh = delta, j, h        # improving move
                if rule == FIRST:
                    stop = True
                    break
        j += 1
    if bj == -1:
        return False                                    # no improving move: local optimum

    L[mach[bj]] = L[mach[bj]] - t[bj]                   # apply the move
    L[bh] = L[bh] + t[bj]
    mach[bj] = bh
    return True

In [14]:
def pcmax_swap_step(n, m, t, mach, L, obj, rule):
    """One step of the swap local search for P||Cmax.
    The move (j, k) exchanges the machines of the jobs j and k;
    obj = CMAX (makespan) or SQ (sum of the squared loads); both are minimized."""
    best_delta = 0                                      # best change of the objective found so far
    bj, bk = -1, -1                                     # best move found so far
    stop = False
    j = 0
    while j < n and not stop:
        for k in range(j + 1, n):
            a, b = mach[j], mach[k]
            if a == b or t[j] == t[k]:
                continue                                # the loads would not change
            delta = objective_change(m, L, a, b, t[j] - t[k], obj)   # t_j - t_k from a to b
            if delta < best_delta:
                best_delta, bj, bk = delta, j, k        # improving move
                if rule == FIRST:
                    stop = True
                    break
        j += 1
    if bj == -1:
        return False                                    # no improving move: local optimum

    a, b = mach[bj], mach[bk]                           # apply the move
    L[a] = L[a] - (t[bj] - t[bk])
    L[b] = L[b] + (t[bj] - t[bk])
    mach[bj] = b
    mach[bk] = a
    return True

## BPP: move and swap

The **move** and **swap steps** for the bin packing problem.

In [15]:
def bpp_move_step(n, w, c, bin, L, obj, rule):
    """One step of the move local search for the BPP.
    bin[i] is the bin of item i, L[b] the load of bin b (L[b] = 0 if bin b is empty), both updated in place;
    obj = NB (number of bins, minimized) or SQ (sum of the squared loads, maximized)."""
    best_gain = 0                                       # best gain found so far
    bi, bb = -1, -1                                     # best move found so far
    stop = False
    i = 0
    while i < n and not stop:
        a = bin[i]                                      # bin of item i
        for b in range(len(L)):
            if b == a or L[b] == 0:
                continue                                # same bin, or empty bin
            if L[b] + w[i] > c:
                continue                                # item i does not fit into bin b
            if obj == SQ:                               # improvement of the objective
                gain = 2 * w[i] * (L[b] - L[a] + w[i])
            else:
                gain = 1 if L[a] == w[i] else 0         # bin a becomes empty
            if gain > best_gain:
                best_gain, bi, bb = gain, i, b          # improving move
                if rule == FIRST:
                    stop = True
                    break
        i += 1
    if bi == -1:
        return False                                    # no improving move: local optimum

    L[bin[bi]] = L[bin[bi]] - w[bi]                     # apply the move
    L[bb] = L[bb] + w[bi]
    bin[bi] = bb
    return True

In [16]:
def bpp_swap_step(n, w, c, bin, L, rule):
    """One step of the swap local search for the BPP, with the auxiliary objective
    sum of the squared loads (maximized); a swap never changes the number of bins.
    The move (i, k) exchanges the bins of the items i and k."""
    best_gain = 0                                       # best gain found so far
    bi, bk = -1, -1                                     # best move found so far
    stop = False
    i = 0
    while i < n and not stop:
        for k in range(i + 1, n):
            a, b = bin[i], bin[k]
            if a == b or w[i] == w[k]:
                continue                                # the loads would not change
            dw = w[i] - w[k]                            # weight transferred from a to b
            if L[b] + dw > c or L[a] - dw > c:
                continue                                # a bin would be overfull
            gain = 2 * dw * (L[b] - L[a] + dw)          # improvement of the objective
            if gain > best_gain:
                best_gain, bi, bk = gain, i, k          # improving move
                if rule == FIRST:
                    stop = True
                    break
        i += 1
    if bi == -1:
        return False                                    # no improving move: local optimum

    a, b, dw = bin[bi], bin[bk], w[bi] - w[bk]          # apply the move
    L[a] = L[a] - dw
    L[b] = L[b] + dw
    bin[bi] = b
    bin[bk] = a
    return True

## KP: flip+swap and penalty flip

The **flip+swap step** and the **penalty flip step** for the KP (the penalty $\alpha = \alpha_{num}/\alpha_{den}$ is a fraction, and all the values are multiplied by $\alpha_{den}$ to work with integers).

In [17]:
def kp_flip_swap_step(n, p, w, c, x, rule):
    """One step of the local search for the KP with the neighbourhood flip + swap,
    restricted to feasible solutions; x[i] = 1 if item i is selected (x is updated in place).
    The flips are examined first, then the swaps (item i out, item k in)."""
    W = 0                                               # weight of the current solution
    for i in range(n):
        W = W + w[i] * x[i]

    best_dp = 0                                         # best profit increase found so far
    bi, bk = -1, -1                                     # best move found so far (bk = -1: flip)
    stop = False
    for i in range(n):                                  # flip of item i
        if stop:
            break
        dp = (1 - 2 * x[i]) * p[i]                      # remove, or add, item i
        dw = (1 - 2 * x[i]) * w[i]
        if W + dw <= c and dp > best_dp:
            best_dp, bi, bk = dp, i, -1                 # improving move
            if rule == FIRST:
                stop = True
    for i in range(n):                                  # swap: item i out, item k in
        if stop:
            break
        if x[i] == 0:
            continue
        for k in range(n):
            if x[k] == 1:
                continue
            dp = p[k] - p[i]
            dw = w[k] - w[i]
            if W + dw <= c and dp > best_dp:
                best_dp, bi, bk = dp, i, k              # improving move
                if rule == FIRST:
                    stop = True
                    break
    if bi == -1:
        return False                                    # no improving move: local optimum

    x[bi] = 1 - x[bi]                                   # apply the flip, or the swap
    if bk != -1:
        x[bk] = 1
    return True

In [18]:
def kp_penalty_flip_step(n, p, w, c, alpha_num, alpha_den, x, rule):
    """One step of the flip local search for the KP with a penalty for the excess weight:
    g(x) = profit(x) - alpha * max(0, weight(x) - c), with alpha = alpha_num / alpha_den.
    Infeasible solutions are allowed; all the values are multiplied by alpha_den (x is updated in place)."""
    P, W = 0, 0                                         # profit and weight of the current solution
    for i in range(n):
        P = P + p[i] * x[i]
        W = W + w[i] * x[i]
    excess = W - c if W > c else 0
    g = alpha_den * P - alpha_num * excess              # alpha_den * g(x)

    best_delta = 0                                      # best increase of alpha_den * g found so far
    bi = -1                                             # best move found so far
    for i in range(n):                                  # flip of item i
        newP = P + (1 - 2 * x[i]) * p[i]
        newW = W + (1 - 2 * x[i]) * w[i]
        new_excess = newW - c if newW > c else 0
        delta = alpha_den * newP - alpha_num * new_excess - g
        if delta > best_delta:
            best_delta, bi = delta, i                   # improving move
            if rule == FIRST:
                break
    if bi == -1:
        return False                                    # no improving move: local optimum

    x[bi] = 1 - x[bi]                                   # apply the flip
    return True

## Multi-start 2-opt

The **multi-start 2-opt** from random tours.

In [19]:
def multi_start_two_opt(n, d, M, seed, rule):
    """Multi-start local search for the TSP: M random tours, each improved by 2-opt.
    The random tours are generated by the Park-Miller generator, starting from seed (an int).
    Returns the pair (length of the best local optimum found, best tour)."""
    seed = [seed]                                       # local copy of the seed, as in C++ (by value)
    best_length = -1                                    # length of the best tour found so far
    best = []
    for s in range(1, M + 1):
        pi = random_tour(n, seed)                       # random starting tour
        local_search(lambda: two_opt_step(n, d, pi, rule))
        length = tour_length(n, d, pi)                  # length of the local optimum
        if best_length == -1 or length < best_length:
            best_length = length                        # new best tour
            best = pi
    return best_length, best

In C++ `MultiStartTwoOpt` fills the output vector `best` and returns the length;
here it returns the pair `(best_length, best)`.

## Execution on the examples of the chapter

Every example of the chapter and every instance of the exercises, with the expected result written in the code: each line shows the result found and `OK` when it is the expected one; the last line counts the checks and the errors.

In [20]:
# runs every example of the lecture notes and every instance of the exercises,
# and compares the values found with those computed by an independent Python engine

errors, checks = 0, 0


def check(name, found, expected):
    global errors, checks
    checks += 1
    ok = (found == expected)
    if not ok:
        errors += 1
    print("%-40s %s" % (name, "OK" if ok else "<-- DIFFERENT"))
    if not ok:
        print("     found    %s\n     expected %s" % (found, expected))


def kp_profit(n, p, x):
    P = 0
    for i in range(n):
        P += p[i] * x[i]
    return str(P)


def kp_pw(n, p, w, x):
    P, W = 0, 0
    for i in range(n):
        P += p[i] * x[i]
        W += w[i] * x[i]
    return "%d/%d" % (P, W)


def bpp_val(n, bin, L):
    S = 0
    for l in L:
        S += l * l
    return "%d/%d" % (number_of_bins(n, bin), S)


def pcm_val(m, L):
    C, S = 0, 0
    for h in range(m):
        if L[h] > C:
            C = L[h]
        S += L[h] * L[h]
    return "%d/%d" % (C, S)


def hist(v):
    v = sorted(v)
    parts = []
    a = 0
    while a < len(v):
        b = a
        while b < len(v) and v[b] == v[a]:
            b += 1
        parts.append("%d:%d" % (v[a], b - a))
        a = b
    return " ".join(parts)


def canon(pi):
    n = len(pi)
    k0 = 0
    while pi[k0] != 0:
        k0 += 1
    c = [pi[(k0 + k) % n] for k in range(n)]
    if c[1] > c[n - 1]:
        c[1:] = c[1:][::-1]
    return tuple(c)


def join(v):
    return " ".join(str(e) for e in v)


def trace(step, value):
    """values of the solution along the local search: while step(): record value()"""
    s = [value()]
    while step():
        s.append(value())
    return " ".join(s)


def trace_vnd(steps, value):
    """values of the solution along the VND: every accepted move records value()"""
    s = [value()]

    def recording(step):
        def f():
            r = step()
            if r:
                s.append(value())
            return r
        return f
    vnd([recording(st) for st in steps])
    return " ".join(s)


RULE = {FIRST: "first", BEST: "best"}

D5 = [[0,5,2,2,9],[5,0,4,3,4],[2,4,0,4,7],[2,3,4,0,7],[9,4,7,7,0]]
D10 = [[0,7,10,12,13,11,7,5,9,4],[7,0,4,7,10,11,7,4,5,3],[10,4,0,4,8,11,7,6,4,6],[12,7,4,0,4,8,6,7,3,9],
       [13,10,8,4,0,5,6,8,5,11],[11,11,11,8,5,0,4,8,7,11],[7,7,7,6,6,4,0,3,4,6],[5,4,6,7,8,8,3,0,4,3],
       [9,5,4,3,5,7,4,4,0,6],[4,3,6,9,11,11,6,3,6,0]]


def points30():
    pts = []
    s0 = [12345]
    for k in range(30):
        a = random_int(s0, 101)
        b = random_int(s0, 101)
        pts.append((a, b))
    return pts


# --- starting solutions
p, w = [10,7,6,4], [5,4,3,3]
check("KP ref: GR", join(greedy_kp(4, p, w, 9, [0,2,1,3])), "1 0 1 0")
check("KP ref: greedy by weight", join(greedy_kp(4, p, w, 9, [2,3,1,0])), "0 0 1 1")
w = [4,4,5,3,2,3]
check("BPP ref: FF", join(first_fit(6, w, 7, [0,1,2,3,4,5])), "0 1 2 0 1 3")
check("BPP ref: NF", join(next_fit(6, w, 7)), "0 1 2 3 3 4")
check("BPP ref: FFD", join(first_fit(6, w, 7, [2,0,1,3,5,4])), "1 2 0 1 0 2")
check("PCmax ref: LPT", join(lpt(7, 3, [5,5,4,4,3,3,3])), "0 1 2 2 0 1 0")
check("TSP ref: NN from city 1", join(nearest_neighbour(5, D5, 0)), "0 2 1 3 4")
check("TSP10: NN from city 1", join(nearest_neighbour(10, D10, 0)), "0 9 1 2 3 8 6 7 4 5")
pts = [(1,1),(3,8),(6,10),(10,9),(13,6),(12,1),(8,3),(5,4),(8,7),(2,5)]
check("TSP10: distances", str(int(euclidean_distances(pts) == D10)), "1")


# --- KP
def kp_fs(p, w, c, x, rule):
    n = len(p)
    return trace(lambda: kp_flip_swap_step(n, p, w, c, x, rule), lambda: kp_profit(n, p, x))


def kp_pen(p, w, c, an, ad, x, rule):
    n = len(p)
    return trace(lambda: kp_penalty_flip_step(n, p, w, c, an, ad, x, rule), lambda: kp_pw(n, p, w, x))


p, w = [10,7,6,4], [5,4,3,3]
check("KP ref from (0,0,1,1) first", kp_fs(p, w, 9, [0,0,1,1], FIRST), "10 14 17")
check("KP ref from (0,0,1,1) best", kp_fs(p, w, 9, [0,0,1,1], BEST), "10 16 17")
for an, ad, exp in [(1, 1, "16/8 23/12 27/15"), (2, 1, "16/8 23/12"), (9, 4, "16/8 23/12 17/9"), (3, 1, "16/8")]:
    for rule in (FIRST, BEST):
        check("KP ref penalty a=%d/%d %s" % (an, ad, RULE[rule]), kp_pen(p, w, 9, an, ad, [1,0,1,0], rule), exp)


# --- BPP
def bpp_run(kind, w, c, bin, obj, rule):
    """kind = 'move', 'swap' or 'vnd' (move then swap, objective obj)"""
    n = len(w)
    L = loads(n, n, bin, w)
    value = lambda: bpp_val(n, bin, L)
    move = lambda: bpp_move_step(n, w, c, bin, L, obj, rule)
    swap = lambda: bpp_swap_step(n, w, c, bin, L, rule)
    if kind == "move":
        return trace(move, value)
    if kind == "swap":
        return trace(swap, value)
    return trace_vnd([move, swap], value)


w = [4,4,5,3,2,3]
for start, sbin, exps in [("FF", [0,1,2,0,1,3], ["4/119", "4/119", "4/119 4/127 3/147", "4/119 4/127 3/147",
                                                  "4/119 4/123", "4/119 4/127", "4/119 4/127 3/147", "4/119 4/127 3/147"]),
                          ("NF", [0,1,2,3,3,4], ["5/91 4/115", "5/91 4/115", "5/91 4/115 4/127 3/147", "5/91 4/115 4/127 3/147",
                                                  "5/91 5/95 5/99", "5/91 5/103", "5/91 4/115 4/127 3/147", "5/91 4/115 4/127 3/147"])]:
    e = 0
    for kind, obj in [("move", NB), ("move", SQ), ("swap", SQ), ("vnd", SQ)]:
        for rule in (FIRST, BEST):
            name = "BPP ref %s %s %s %s" % (start, kind, "nb" if obj == NB else "sq", RULE[rule])
            check(name, bpp_run(kind, w, 7, list(sbin), obj, rule), exps[e])
            e += 1


# --- TSP
def tsp_run(kind, d, pi, rule):
    """kind = '2opt', 'or' or 'vnd' (2-opt then Or-opt)"""
    n = len(pi)
    value = lambda: str(tour_length(n, d, pi))
    two = lambda: two_opt_step(n, d, pi, rule)
    orr = lambda: or_opt_step(n, d, pi, rule)
    if kind == "2opt":
        return trace(two, value)
    if kind == "or":
        return trace(orr, value)
    return trace_vnd([two, orr], value)


check("TSP ref NN 2-opt first", tsp_run("2opt", D5, [0,2,1,3,4], FIRST), "25 22 18")
check("TSP ref NN 2-opt best", tsp_run("2opt", D5, [0,2,1,3,4], BEST), "25 19 18")
for kind, exps in [("2opt", ["49 47 46 43 41", "49 44 40 39"]), ("or", ["49 47 46 40 39", "49 40 39"]),
                   ("vnd", ["49 47 46 43 41 39", "49 44 40 39"])]:
    for e, rule in enumerate((FIRST, BEST)):
        check("TSP10 NN %s %s" % (kind, RULE[rule]), tsp_run(kind, D10, [0,9,1,2,3,8,6,7,4,5], rule), exps[e])
check("TSP10 Or-opt from 41 first", tsp_run("or", D10, [0,9,1,2,3,4,5,6,8,7], FIRST), "41 39")
check("TSP10 Or-opt from 41 best", tsp_run("or", D10, [0,9,1,2,3,4,5,6,8,7], BEST), "41 39")


# --- P||Cmax
def pcm_run(kind, t, m, mach, obj, rule):
    """kind = 'move', 'swap' or 'vnd' (move then swap)"""
    n = len(t)
    L = loads(n, m, mach, t)
    value = lambda: pcm_val(m, L)
    move = lambda: pcmax_move_step(n, m, t, mach, L, obj, rule)
    swap = lambda: pcmax_swap_step(n, m, t, mach, L, obj, rule)
    if kind == "move":
        return trace(move, value)
    if kind == "swap":
        return trace(swap, value)
    return trace_vnd([move, swap], value)


OBJ = {CMAX: "cmax", SQ: "sq"}
t = [5,5,4,4,3,3,3]
for kind, exp in [("move", "11/249"), ("swap", "11/249 10/245 9/243"), ("vnd", "11/249 10/245 9/243")]:
    for obj in (CMAX, SQ):
        for rule in (FIRST, BEST):
            check("PCmax ref LPT %s %s %s" % (kind, OBJ[obj], RULE[rule]), pcm_run(kind, t, 3, [0,1,2,2,0,1,0], obj, rule), exp)
t12 = [11,10,9,8,7,7,6,5,4,4,3,2]
for kind in ("move", "swap"):
    for rule in (FIRST, BEST):
        check("PCM12 LPT %s cmax %s" % (kind, RULE[rule]), pcm_run(kind, t12, 4, [0,1,2,3,3,2,1,0,3,0,1,2], CMAX, rule), "20/1446")


def pcm_starts(kind, obj):
    """100 random assignments of the 12 jobs to 4 machines, each improved by move, swap or VND (best improvement)"""
    t, n, m = t12, 12, 4
    seed = [1]
    res = []
    for s in range(100):
        mach = random_assignment(n, m, seed)
        L = loads(n, m, mach, t)
        move = lambda: pcmax_move_step(n, m, t, mach, L, obj, BEST)
        swap = lambda: pcmax_swap_step(n, m, t, mach, L, obj, BEST)
        if kind == "move":
            local_search(move)
        elif kind == "swap":
            local_search(swap)
        else:
            vnd([move, swap])
        C = 0
        for h in range(m):
            if L[h] > C:
                C = L[h]
        res.append(C)
    return hist(res)


check("PCM12 100 starts move cmax", pcm_starts("move", CMAX), "19:3 20:48 21:29 22:12 23:3 25:2 26:2 33:1")
check("PCM12 100 starts move sq", pcm_starts("move", SQ), "19:4 20:61 21:27 22:8")
check("PCM12 100 starts swap cmax", pcm_starts("swap", CMAX), "19:7 20:10 21:6 22:15 23:11 24:12 25:10 26:5 27:3 28:8 29:2 30:2 31:5 33:1 34:1 38:2")
check("PCM12 100 starts swap sq", pcm_starts("swap", SQ), "19:20 20:15 22:17 23:14 24:11 26:6 28:10 31:4 33:1 38:2")
check("PCM12 100 starts vnd cmax", pcm_starts("vnd", CMAX), "19:15 20:52 21:17 22:8 23:3 25:2 26:2 33:1")
check("PCM12 100 starts vnd sq", pcm_starts("vnd", SQ), "19:68 20:32")


def bpp_starts(kind, obj):
    """100 first fit solutions with random item orders, each improved by move, swap or VND (best improvement)"""
    w, n, c = [7,6,6,5,5,4,4,4,3,3,2,1], 12, 10
    seed = [1]
    res = []
    for s in range(100):
        bin = first_fit(n, w, c, random_permutation(n, seed))
        L = loads(n, n, bin, w)
        move = lambda: bpp_move_step(n, w, c, bin, L, obj, BEST)
        swap = lambda: bpp_swap_step(n, w, c, bin, L, BEST)
        if kind == "move":
            local_search(move)
        elif kind == "swap":
            local_search(swap)
        elif kind == "vnd":
            vnd([move, swap])
        res.append(number_of_bins(n, bin))
    return hist(res)


check("BPP12 100 starts start nb", bpp_starts("start", NB), "5:14 6:85 7:1")
check("BPP12 100 starts move nb", bpp_starts("move", NB), "5:14 6:85 7:1")
check("BPP12 100 starts move sq", bpp_starts("move", SQ), "5:76 6:24")
check("BPP12 100 starts swap sq", bpp_starts("swap", SQ), "5:14 6:85 7:1")
check("BPP12 100 starts vnd sq", bpp_starts("vnd", SQ), "5:89 6:11")


def tsp_starts(d, kind, rule):
    """lengths of the local optima from 100 random tours (2-opt, Or-opt or VND), and the set of distinct optima"""
    n = len(d)
    seed = [1]
    res = []
    S = set()
    for s in range(100):
        pi = random_tour(n, seed)
        two = lambda: two_opt_step(n, d, pi, rule)
        orr = lambda: or_opt_step(n, d, pi, rule)
        if kind == "2opt":
            local_search(two)
        elif kind == "or":
            local_search(orr)
        else:
            vnd([two, orr])
        res.append(tour_length(n, d, pi))
        S.add(canon(pi))
    return hist(res), len(S)


check("TSP10 100 starts 2-opt first", tsp_starts(D10, "2opt", FIRST)[0], "39:46 41:54")
check("TSP10 MultiStartTwoOpt first", str(multi_start_two_opt(10, D10, 100, 1, FIRST)[0]), "39")
check("TSP10 100 starts 2-opt best", tsp_starts(D10, "2opt", BEST)[0], "39:87 41:8 43:5")
check("TSP10 MultiStartTwoOpt best", str(multi_start_two_opt(10, D10, 100, 1, BEST)[0]), "39")
D30 = euclidean_distances(points30())
check("TSP30 100 starts 2-opt first", tsp_starts(D30, "2opt", FIRST)[0], "478:14 479:26 480:9 481:6 482:2 484:2 485:1 486:1 488:2 489:1 490:1 493:2 494:2 495:4 496:4 497:5 501:1 502:3 504:1 506:1 508:2 510:1 514:2 515:2 517:1 518:1 519:1 520:1 537:1")
check("TSP30 MultiStartTwoOpt first", str(multi_start_two_opt(30, D30, 100, 1, FIRST)[0]), "478")
check("TSP30 100 starts 2-opt best", tsp_starts(D30, "2opt", BEST)[0], "478:12 479:34 480:5 481:8 482:1 484:2 485:1 486:1 487:3 489:1 490:1 491:2 492:1 493:2 494:1 495:6 497:1 500:1 501:4 502:2 503:3 508:2 512:2 513:1 515:1 520:1 567:1")
check("TSP30 MultiStartTwoOpt best", str(multi_start_two_opt(30, D30, 100, 1, BEST)[0]), "478")

# --- exercises: KP
p, w = [10,7,13,14,8], [2,3,8,9,7]
check("EX_KP_1 from GR first", kp_fs(p, w, 18, [1,1,1,0,0], FIRST), "30 31 32")
check("EX_KP_1 from GR best", kp_fs(p, w, 18, [1,1,1,0,0], BEST), "30 31 32")
check("EX_KP_1 from greedy by profit best", kp_fs(p, w, 18, [0,0,1,1,0], BEST), "27")
p, w = [8,9,14,11,9], [3,5,9,9,8]
check("EX_KP_2 from GR first", kp_fs(p, w, 13, [1,1,0,0,0], FIRST), "17 18")
check("EX_KP_2 from GR best", kp_fs(p, w, 13, [1,1,0,0,0], BEST), "17 22")
p, w = [14,7,10,8,6], [4,4,7,7,9]
for an, exp in [(1, "21/8 31/15 39/22"), (2, "21/8 31/15 24/11"), (3, "21/8"), (15, "21/8")]:
    check("EX_KP_3 penalty a=%d best" % an, kp_pen(p, w, 11, an, 1, [1,1,0,0,0], BEST), exp)

# --- exercises: BPP
w, b0 = [2,3,2,7,4,7,4], [0,0,0,1,2,3,2]
check("EX_BPP_1 FF move nb first", bpp_run("move", w, 10, list(b0), NB, FIRST), "4/211")
check("EX_BPP_1 FF move nb best", bpp_run("move", w, 10, list(b0), NB, BEST), "4/211")
check("EX_BPP_1 FF move sq first", bpp_run("move", w, 10, list(b0), SQ, FIRST), "4/211 4/219 4/223 4/253 3/281")
check("EX_BPP_1 FF move sq best", bpp_run("move", w, 10, list(b0), SQ, BEST), "4/211 4/229 4/253 3/281")
w, b0 = [2,3,4,5,4,2], [0,0,0,1,1,2]
check("EX_BPP_2 FF move sq best", bpp_run("move", w, 10, list(b0), SQ, BEST), "3/166")
check("EX_BPP_2 FF swap sq best", bpp_run("swap", w, 10, list(b0), SQ, BEST), "3/166 3/168")
check("EX_BPP_2 FF vnd sq best", bpp_run("vnd", w, 10, list(b0), SQ, BEST), "3/166 3/168 2/200")
w, b0 = [5,3,3,2,8,4,4], [0,0,1,0,2,1,3]
check("EX_BPP_3 FF move sq first", bpp_run("move", w, 10, list(b0), SQ, FIRST), "4/229 4/237")
check("EX_BPP_3 FF move sq best", bpp_run("move", w, 10, list(b0), SQ, BEST), "4/229 4/237")
check("EX_BPP_3 FF vnd sq first", bpp_run("vnd", w, 10, list(b0), SQ, FIRST), "4/229 4/237")
check("EX_BPP_3 FF vnd sq best", bpp_run("vnd", w, 10, list(b0), SQ, BEST), "4/229 4/237")

# --- exercises: TSP
EX_TSP = [("EX_TSP_1", [[0,7,6,3,4,6],[7,0,6,5,7,4],[6,6,0,5,5,2],[3,5,5,0,5,3],[4,7,5,5,0,4],[6,4,2,3,4,0]], [0,3,5,2,4,1],
           ["27 26 23", "27 25 24 23", "27 25 23", "27 23", "27 26 23", "27 25 24 23"]),
          ("EX_TSP_2", [[0,8,5,6,5,6],[8,0,6,7,7,6],[5,6,0,4,8,2],[6,7,4,0,6,5],[5,7,8,6,0,7],[6,6,2,5,7,0]], [0,2,5,3,4,1],
           ["33 32 31 30", "33 31", "33 31 30", "33 31 30", "33 32 31 30", "33 31 30"]),
          ("EX_TSP_3", [[0,7,8,8,9,9],[7,0,7,4,9,8],[8,7,0,7,3,9],[8,4,7,0,5,6],[9,9,3,5,0,7],[9,8,9,6,7,0]], [0,1,3,4,2,5],
           ["37", "37", "37 35", "37 35", "37 35", "37 35"])]
for name, d, pi0, exps in EX_TSP:
    e = 0
    for kind in ("2opt", "or", "vnd"):
        for rule in (FIRST, BEST):
            check("%s NN %s %s" % (name, kind, RULE[rule]), tsp_run(kind, d, list(pi0), rule), exps[e])
            e += 1

# --- exercises: P||Cmax
t, m0 = [5,7,3,4,4,9], [0,1,2,2,0,1]
check("EX_PCM_1 LIST move cmax first", pcm_run("move", t, 3, list(m0), CMAX, FIRST), "16/386 14/358 12/346")
check("EX_PCM_1 LIST move cmax best", pcm_run("move", t, 3, list(m0), CMAX, BEST), "16/386 14/358 12/346")
t, m0 = [8,4,6,7,5,9,8], [1,0,2,1,0,0,2]
check("EX_PCM_2 LPT move cmax best", pcm_run("move", t, 3, list(m0), CMAX, BEST), "18/745")
check("EX_PCM_2 LPT swap cmax best", pcm_run("swap", t, 3, list(m0), CMAX, BEST), "18/745 17/741 16/737")
t, m0 = [5,5,7,9,2,7], [0,1,2,0,1,1]
check("EX_PCM_3 LIST move cmax best", pcm_run("move", t, 3, list(m0), CMAX, BEST), "14/441")
check("EX_PCM_3 LIST move sq best", pcm_run("move", t, 3, list(m0), SQ, BEST), "14/441 14/421 12/409")

# --- multi-start statistics
check("TSP10 100 starts or first", tsp_starts(D10, "or", FIRST)[0], "39:72 43:26 44:2")
check("TSP10 100 starts or best", tsp_starts(D10, "or", BEST)[0], "39:77 43:22 44:1")
check("TSP10 100 starts vnd first", tsp_starts(D10, "vnd", FIRST)[0], "39:100")
check("TSP10 100 starts vnd best", tsp_starts(D10, "vnd", BEST)[0], "39:100")
check("TSP10 distinct 2-opt local optima first", str(tsp_starts(D10, "2opt", FIRST)[1]), "2")
check("TSP10 distinct 2-opt local optima best", str(tsp_starts(D10, "2opt", BEST)[1]), "3")
check("TSP30 distinct 2-opt local optima first", str(tsp_starts(D30, "2opt", FIRST)[1]), "59")
check("TSP30 distinct 2-opt local optima best", str(tsp_starts(D30, "2opt", BEST)[1]), "54")
seed = [1]
tot, mn, mx = 0, 1000000, 0
for s in range(100):
    pi = random_tour(30, seed)
    Lt = tour_length(30, D30, pi)
    tot += Lt
    if Lt < mn:
        mn = Lt
    if Lt > mx:
        mx = Lt
check("TSP30 random tours: sum min max", "%d %d %d" % (tot, mn, mx), "176873 1364 2080")

print("\n%d checks, %d errors" % (checks, errors))

KP ref: GR                               OK
KP ref: greedy by weight                 OK
BPP ref: FF                              OK
BPP ref: NF                              OK
BPP ref: FFD                             OK
PCmax ref: LPT                           OK
TSP ref: NN from city 1                  OK
TSP10: NN from city 1                    OK
TSP10: distances                         OK
KP ref from (0,0,1,1) first              OK
KP ref from (0,0,1,1) best               OK
KP ref penalty a=1/1 first               OK
KP ref penalty a=1/1 best                OK
KP ref penalty a=2/1 first               OK
KP ref penalty a=2/1 best                OK
KP ref penalty a=9/4 first               OK
KP ref penalty a=9/4 best                OK
KP ref penalty a=3/1 first               OK
KP ref penalty a=3/1 best                OK
BPP ref FF move nb first                 OK
BPP ref FF move nb best                  OK
BPP ref FF move sq first                 OK
BPP ref FF move sq best         

## Try it yourself

Change the data in the cell below and run it again.

In [21]:
# change the data below and run the cell again
points = [(0, 0), (9, 1), (2, 7), (8, 8), (4, 3), (1, 4), (7, 4), (5, 9)]   # TSP: coordinates of the cities
t = [7, 6, 5, 5, 4, 4, 3, 2]                   # P||Cmax: processing times of the jobs
m = 3                                          # P||Cmax: number of machines
rule = BEST                                    # FIRST or BEST

def show(v):
    return " ".join(str(e) for e in v)


n = len(points)
d = euclidean_distances(points)
seed = [1]
pi = random_tour(n, seed)
print("TSP random tour: %s  length %d" % (show(pi), tour_length(n, d, pi)))
moves = local_search(lambda: or_opt_step(n, d, pi, rule))
print("after Or-opt (%d moves): %s  length %d" % (moves, show(pi), tour_length(n, d, pi)))
moves = vnd([lambda: two_opt_step(n, d, pi, rule), lambda: or_opt_step(n, d, pi, rule)])
print("after VND 2-opt + Or-opt (%d moves): %s  length %d" % (moves, show(pi), tour_length(n, d, pi)))
length, best = multi_start_two_opt(n, d, 20, 1, rule)
print("multi-start 2-opt, 20 starts: %s  length %d" % (show(best), length))

nj = len(t)
mach = random_assignment(nj, m, seed)
L = loads(nj, m, mach, t)
print("P||Cmax random assignment: %s  loads %s" % (show(mach), show(L)))
moves = vnd([lambda: pcmax_move_step(nj, m, t, mach, L, SQ, rule),
             lambda: pcmax_swap_step(nj, m, t, mach, L, SQ, rule)])
print("after VND move + swap (%d moves): %s  loads %s" % (moves, show(mach), show(L)))
mach = lpt(nj, m, t)
print("LPT, for comparison: %s  loads %s" % (show(mach), show(loads(nj, m, mach, t))))

TSP random tour: 0 3 6 5 2 4 7 1  length 52
after Or-opt (2 moves): 0 5 2 7 3 6 1 4  length 32
after VND 2-opt + Or-opt (0 moves): 0 5 2 7 3 6 1 4  length 32
multi-start 2-opt, 20 starts: 0 4 1 6 3 7 2 5  length 32
P||Cmax random assignment: 0 2 2 2 1 1 2 0  loads 9 8 19
after VND move + swap (4 moves): 0 1 0 2 2 1 2 1  loads 12 12 12
LPT, for comparison: 0 1 2 2 1 0 1 2  loads 11 13 12


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/combinatorial-optimization/blob/main/LICENSE-CODE).